# 02 — Generate a 100-flight JSBSim skill-recognition dataset

This notebook creates a small, reproducible **1-v-1 F-16** dataset for neural-network pipeline tests. Each aircraft is a separate JSBSim flight-dynamics instance. A seeded `StochasticSkillManager` varies geometry and skill schedules, while BVR Sim's versioned `SkillManager` supplies the commanded-skill contract and labels.

Outputs follow the revised plan's separation of target-centric observable features and privileged labels, include multi-head commanded/native/derived labels and transition timing, and write one Tacview 2.1 `.acmi` replay per flight.

> This is synthetic research data, not tactical or flight-control guidance. Open-loop control mappings are deliberately bounded and should be visually quality-checked before model training. Weapons are disabled.


## 1. Imports and reproducible build configuration

The integration timestep is finer than the logging interval. `SAMPLE_DT_S=0.1` guarantees samples every 0.1 seconds of game time; changing it to a larger value is rejected. Set `BVR_DATASET_FLIGHTS` for a shorter smoke run.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import subprocess
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

import jsbsim
import numpy as np
import pandas as pd
import yaml

from bvr_behavior_prediction.data.labels import kinematic_labels
from bvr_behavior_prediction.data.observable_columns import MODEL_FEATURE_COLUMNS
from bvr_behavior_prediction.data.privileged_columns import PRIVILEGED_COLUMNS
from bvr_behavior_prediction.data.schema import TRAJECTORY_COLUMNS, validate_columns
from bvr_behavior_prediction.generation.dataset_builder import DatasetBuilder
from bvr_behavior_prediction.generation.manifest import DatasetManifest
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
BVR_SOURCE = REPO_ROOT / "bvr_sim_source"
assert BVR_SOURCE.exists(), "Expected the bundled bvr_sim_source checkout"
sys.path.insert(0, str(BVR_SOURCE))
from bvr_sim.agents.skill_manager import SkillManager

OUTPUT_DIR = Path(os.getenv(
    "BVR_NOTEBOOK_OUTPUT", REPO_ROOT / "artifacts/datasets/bvr_f16_1v1_jsbsim_skills_v001"
))
N_FLIGHTS = int(os.getenv("BVR_DATASET_FLIGHTS", "100"))
BASE_SEED = int(os.getenv("BVR_DATASET_SEED", "20260911"))
DURATION_S = float(os.getenv("BVR_DATASET_DURATION_S", "45"))
SIM_DT_S = 1 / 60
SAMPLE_DT_S = 0.1
assert 0 < SAMPLE_DT_S <= 0.1
assert round(SAMPLE_DT_S / SIM_DT_S) * SIM_DT_S == SAMPLE_DT_S
JSBSIM_ROOT = os.getenv("JSBSIM_ROOT") or None
MODEL = os.getenv("JSBSIM_MODEL", "f16")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "acmi").mkdir(exist_ok=True)
print({"flights": N_FLIGHTS, "duration_s": DURATION_S, "sample_dt_s": SAMPLE_DT_S})


## 2. Stochastic scenario and skill manager

Sampling is stratified by primary target skill so all six initial tactical classes are represented. Within each class, start range, lateral offset, altitude, speed, headings, switch times, and secondary skill vary. The exact schedule is retained as privileged provenance.

In [ ]:
TACTICAL_SKILLS = {
    "MAINTAIN": "maintain_heading",
    "PURSUE": "pursue_target",
    "BEAM": "beam_target_left",
    "CRANK_LEFT": "crank_target_left",
    "CRANK_RIGHT": "crank_target_right",
    "EXTEND": "extend",
}

@dataclass(frozen=True)
class FlightScenario:
    seed: int
    range_nm: float
    lateral_offset_nm: float
    observer_altitude_ft: float
    altitude_difference_m: float
    observer_speed_fps: float
    speed_difference_ms: float
    observer_heading_deg: float
    heading_difference_deg: float
    switch_time_s: float
    primary_label: str
    secondary_label: str


class StochasticSkillManager:
    """Seeded sampler around BVR Sim's versioned SkillManager catalogue."""

    def __init__(self, seed: int):
        self.rng = random.Random(seed)
        self.catalogue = SkillManager()

    def sample(self, flight_index: int) -> FlightScenario:
        labels = tuple(TACTICAL_SKILLS)
        primary = labels[flight_index % len(labels)]  # stratified coverage
        secondary = self.rng.choice([label for label in labels if label != primary])
        return FlightScenario(
            seed=BASE_SEED + flight_index,
            range_nm=self.rng.uniform(12.0, 42.0),
            lateral_offset_nm=self.rng.uniform(-5.0, 5.0),
            observer_altitude_ft=self.rng.uniform(14_000.0, 28_000.0),
            altitude_difference_m=self.rng.uniform(-2_000.0, 2_000.0),
            observer_speed_fps=self.rng.uniform(650.0, 900.0),
            speed_difference_ms=self.rng.uniform(-60.0, 60.0),
            observer_heading_deg=self.rng.uniform(0.0, 360.0),
            heading_difference_deg=self.rng.uniform(25.0, 200.0),
            switch_time_s=self.rng.uniform(18.0, 30.0),
            primary_label=primary,
            secondary_label=secondary,
        )

    def active(self, scenario: FlightScenario, time_s: float):
        label = scenario.primary_label if time_s < scenario.switch_time_s else scenario.secondary_label
        name = TACTICAL_SKILLS[label]
        params = {"timeout_s": DURATION_S}
        if name == "maintain_heading":
            params["heading_deg"] = scenario.observer_heading_deg
        # Creating through SkillManager validates parameters and preserves its contract version.
        return label, self.catalogue.create_skill(name, params), self.catalogue.get_contract(name)

stochastic_manager = StochasticSkillManager(BASE_SEED)
scenarios = [stochastic_manager.sample(i) for i in range(N_FLIGHTS)]
pd.DataFrame(map(asdict, scenarios)).groupby("primary_label").size()


## 3. JSBSim aircraft wrapper and bounded skill controls

Every aircraft receives identical model/version handling. Target control depends only on the active labelled skill and current geometry. Observer control remains a stable `MAINTAIN` reference. Skill commands are evaluated on every logged step; the controller mapping is intentionally explicit rather than treating normalized FCS commands as BVR Sim native action bins.

In [ ]:
FT_TO_M = 0.3048
FPS_TO_MPS = 0.3048
EARTH_RADIUS_M = 6_378_137.0


def clamp(value, low, high):
    return max(low, min(high, value))


def angle_error_deg(target, current):
    return (target - current + 180.0) % 360.0 - 180.0


class JSBSimAircraft:
    def __init__(self, initial):
        self.fdm = jsbsim.FGFDMExec(JSBSIM_ROOT)
        self.fdm.set_debug_level(0)
        self.fdm.set_dt(SIM_DT_S)
        if not self.fdm.load_model(MODEL):
            raise RuntimeError(f"Could not load JSBSim model {MODEL!r}")
        for key, value in initial.items():
            self.fdm[key] = value
        if not self.fdm.run_ic():
            raise RuntimeError("JSBSim rejected initial conditions")

    def state(self):
        f = self.fdm
        return {
            "latitude_deg": float(f["position/lat-gc-deg"]),
            "longitude_deg": float(f["position/long-gc-deg"]),
            "z": float(f["position/h-sl-ft"]) * FT_TO_M,
            "heading": math.radians(float(f["attitude/psi-deg"])),
            "pitch": math.radians(float(f["attitude/theta-deg"])),
            "roll": math.radians(float(f["attitude/phi-deg"])),
            "speed": float(f["velocities/vtrue-fps"]) * FPS_TO_MPS,
            "vz": float(f["velocities/v-down-fps"]) * -FPS_TO_MPS,
        }

    def step(self, controls):
        self.fdm["fcs/aileron-cmd-norm"] = controls[0]
        self.fdm["fcs/elevator-cmd-norm"] = controls[1]
        self.fdm["fcs/rudder-cmd-norm"] = controls[2]
        self.fdm["fcs/throttle-cmd-norm"] = controls[3]
        if not self.fdm.run():
            raise RuntimeError("JSBSim stopped unexpectedly")


def target_controls(label, own, opponent):
    bearing = math.degrees(math.atan2(opponent["y"] - own["y"], opponent["x"] - own["x"])) % 360
    headings = {
        "MAINTAIN": math.degrees(own["heading"]), "PURSUE": bearing,
        "BEAM": bearing - 90, "CRANK_LEFT": bearing - 45,
        "CRANK_RIGHT": bearing + 45, "EXTEND": bearing + 180,
    }
    heading_error = angle_error_deg(headings[label], math.degrees(own["heading"]))
    # Command bank, rather than holding aileron until the desired heading is crossed.
    # Roll and pitch feedback keep long manoeuvres inside the F-16 flight envelope.
    desired_roll_deg = clamp(heading_error, -35.0, 35.0)
    roll_error_deg = desired_roll_deg - math.degrees(own["roll"])
    aileron = clamp(roll_error_deg / 45.0, -0.35, 0.35)
    desired_pitch_deg = 2.0 if label in {"PURSUE", "CRANK_LEFT", "CRANK_RIGHT"} else 0.0
    elevator = clamp((math.degrees(own["pitch"]) - desired_pitch_deg) / 30.0, -0.15, 0.15)
    throttle = 0.86 if label in {"PURSUE", "EXTEND"} else 0.78
    return aileron, elevator, 0.0, throttle


def native_bins(controls):
    aileron, elevator, _rudder, throttle = controls
    return (
        int(np.clip(round(7 + 7 * aileron), 0, 14)),
        int(np.clip(round(7 - 7 * elevator), 0, 14)),
        int(np.clip(round(8 * throttle), 0, 8)), 0,
    )


## 4. Relative geometry, hierarchical labels, and Tacview writer

Coordinates use a per-flight local ENU approximation for learning while ACMI retains geodetic coordinates. Derived kinematic heads describe actual motion; `commanded_skill` and native action bins remain privileged.

In [ ]:
def add_local_state(state, lat0, lon0):
    state = dict(state)
    state["x"] = math.radians(state["longitude_deg"] - lon0) * EARTH_RADIUS_M * math.cos(math.radians(lat0))
    state["y"] = math.radians(state["latitude_deg"] - lat0) * EARTH_RADIUS_M
    horizontal = math.sqrt(max(0.0, state["speed"] ** 2 - state["vz"] ** 2))
    state["vx"] = horizontal * math.sin(state["heading"])
    state["vy"] = horizontal * math.cos(state["heading"])
    return state


def relative_features(observer, target, previous, sample_dt):
    dx, dy, dz = target["x"]-observer["x"], target["y"]-observer["y"], target["z"]-observer["z"]
    c, s = math.cos(observer["heading"]), math.sin(observer["heading"])
    dvx, dvy, dvz = target["vx"]-observer["vx"], target["vy"]-observer["vy"], target["vz"]-observer["vz"]
    rng = math.sqrt(dx*dx + dy*dy + dz*dz)
    bearing = math.atan2(dx, dy)
    los_az = math.atan2(dy, dx)
    los_el = math.atan2(dz, math.hypot(dx, dy))
    turn_rate = climb_rate = acceleration = los_rate = 0.0
    if previous:
        turn_rate = math.radians(angle_error_deg(math.degrees(target["heading"]), math.degrees(previous["heading"]))) / sample_dt
        climb_rate = (target["z"] - previous["z"]) / sample_dt
        acceleration = (target["speed"] - previous["speed"]) / sample_dt
        los_rate = angle_error_deg(math.degrees(los_az), math.degrees(previous["los_azimuth"])) * math.pi / 180 / sample_dt
    return {
        "rel_x_body": c*dx+s*dy, "rel_y_body": -s*dx+c*dy, "rel_z_body": dz,
        "rel_vx_body": c*dvx+s*dvy, "rel_vy_body": -s*dvx+c*dvy, "rel_vz_body": dvz,
        "range": rng, "range_rate": (dx*dvx+dy*dvy+dz*dvz)/max(rng, 1),
        "relative_bearing": angle_error_deg(math.degrees(bearing), math.degrees(observer["heading"])) * math.pi/180,
        "relative_heading": angle_error_deg(math.degrees(target["heading"]), math.degrees(observer["heading"])) * math.pi/180,
        "relative_altitude": dz, "relative_speed": target["speed"]-observer["speed"],
        "target_aspect": angle_error_deg(math.degrees(bearing)+180, math.degrees(target["heading"])) * math.pi/180,
        "angle_off": angle_error_deg(math.degrees(target["heading"]), math.degrees(observer["heading"])) * math.pi/180,
        "los_azimuth": los_az, "los_elevation": los_el, "los_rate": los_rate,
        "target_turn_rate": turn_rate, "target_climb_rate": climb_rate, "target_acceleration": acceleration,
    }


def write_acmi(path, frames):
    with path.open("w", encoding="utf-8-sig") as fh:
        fh.write("FileType=text/acmi/tacview\nFileVersion=2.1\n")
        fh.write("0,ReferenceTime=2026-01-01T00:00:00Z,Title=JSBSim skill dataset flight\n")
        for time_s, observer, target in frames:
            fh.write(f"#{time_s:.2f}\n")
            for object_id, name, color, state in ((1,"Observer","Red",observer),(2,"Target","Blue",target)):
                fh.write(f'{object_id},T={state["longitude_deg"]:.8f}|{state["latitude_deg"]:.8f}|{state["z"]:.2f}|{math.degrees(state["roll"]):.2f}|{math.degrees(state["pitch"]):.2f}|{math.degrees(state["heading"]):.2f},Name={name},Color={color},Type=Air+FixedWing\n')


## 5. Run approximately 100 flights

Samples are recorded at exactly 10 Hz. A fresh pair of FDM instances is constructed per flight. The first sample is at game time 0.0; no wall-clock timestamps are used.

In [ ]:
def initial_conditions(scenario, target=False):
    # At this latitude, offsets are converted to geodetic initial positions.
    lat0, lon0 = 37.62, -122.38
    if target:
        north_m = scenario.range_nm * 1852.0
        east_m = scenario.lateral_offset_nm * 1852.0
        lat = lat0 + math.degrees(north_m / EARTH_RADIUS_M)
        lon = lon0 + math.degrees(east_m / (EARTH_RADIUS_M * math.cos(math.radians(lat0))))
        altitude_ft = scenario.observer_altitude_ft + scenario.altitude_difference_m / FT_TO_M
        speed_fps = scenario.observer_speed_fps + scenario.speed_difference_ms / FPS_TO_MPS
        heading = (scenario.observer_heading_deg + scenario.heading_difference_deg) % 360
    else:
        lat, lon, altitude_ft, speed_fps, heading = lat0, lon0, scenario.observer_altitude_ft, scenario.observer_speed_fps, scenario.observer_heading_deg
    return {"ic/lat-gc-deg":lat,"ic/long-gc-deg":lon,"ic/h-sl-ft":altitude_ft,"ic/psi-true-deg":heading,
            "ic/u-fps":speed_fps,"ic/v-fps":0.0,"ic/w-fps":0.0,"ic/p-rad_sec":0.0,"ic/q-rad_sec":0.0,"ic/r-rad_sec":0.0}


def run_flight(index, scenario):
    random.seed(scenario.seed); np.random.seed(scenario.seed)
    observer_fdm = JSBSimAircraft(initial_conditions(scenario, False))
    target_fdm = JSBSimAircraft(initial_conditions(scenario, True))
    episode_id = f"jsbsim-{index:04d}"
    rows, acmi_frames = [], []
    previous_target = previous_rel = None
    sample_stride = round(SAMPLE_DT_S / SIM_DT_S)
    n_samples = round(DURATION_S / SAMPLE_DT_S) + 1
    skill_version = None
    for sample_index in range(n_samples):
        time_s = sample_index * SAMPLE_DT_S
        observer_raw, target_raw = observer_fdm.state(), target_fdm.state()
        lat0, lon0 = observer_raw["latitude_deg"], observer_raw["longitude_deg"]
        observer = add_local_state(observer_raw, lat0, lon0)
        target = add_local_state(target_raw, lat0, lon0)
        label, skill, contract = stochastic_manager.active(scenario, time_s)
        skill_version = contract["version"]
        skill.execute({"time":time_s,"self_status":{"position":{"heading_deg":math.degrees(target["heading"]),"altitude_m":target["z"]},"performance":{"speed_mps":target["speed"]}}})
        controls = target_controls(label, target, observer)
        bins = native_bins(controls)
        rel = relative_features(observer, target, ({**previous_target, **previous_rel} if previous_target else None), SAMPLE_DT_S)
        lateral, vertical, energy = kinematic_labels(rel["target_turn_rate"], rel["target_climb_rate"], rel["target_acceleration"])
        row = {"episode_id":episode_id,"perspective_id":f"{episode_id}:observer","step":sample_index,"time_s":time_s,
               "observer_id":"observer","target_id":"target","observer_aircraft_type":"F16","target_aircraft_type":"F16"}
        for prefix, state in (("observer",observer),("target",target)):
            row.update({f"{prefix}_{key}":state[key] for key in ("x","y","z","vx","vy","vz","heading","pitch","roll","speed")})
        row.update(rel)
        row.update({"track_valid":True,"track_age":0.0,"sensor_mode":"truth","target_policy":"StochasticSkillManager",
                    "target_skill":label,"target_skill_id":TACTICAL_SKILLS[label],"commanded_skill":label,
                    "target_action_heading_bin":bins[0],"target_action_altitude_bin":bins[1],"target_action_speed_bin":bins[2],"target_action_fire":bins[3],
                    "lateral_label":lateral.name,"vertical_label":vertical.name,"energy_label":energy.name,"tactical_label":label,
                    "transition_flag":int(sample_index > 0 and rows[-1]["target_skill"] != label),
                    "time_since_skill_change":time_s if time_s < scenario.switch_time_s else time_s-scenario.switch_time_s,
                    "time_to_next_skill_change":max(0.0, scenario.switch_time_s-time_s) if time_s < scenario.switch_time_s else float("inf"),
                    "target_next_skill":scenario.secondary_label if time_s < scenario.switch_time_s else label})
        for horizon in (2,4,8): row[f"transition_within_{horizon}s"] = int(0 < row["time_to_next_skill_change"] <= horizon)
        rows.append(row); acmi_frames.append((time_s, observer_raw, target_raw))
        previous_target, previous_rel = target, rel
        if sample_index < n_samples - 1:
            for _ in range(sample_stride):
                observer_fdm.step(target_controls("MAINTAIN", observer, target)); target_fdm.step(controls)
    write_acmi(OUTPUT_DIR / "acmi" / f"{episode_id}.txt.acmi", acmi_frames)
    return rows, skill_version

trajectory_rows, episode_rows = [], []
for index, scenario in enumerate(scenarios):
    rows, skill_version = run_flight(index, scenario)
    trajectory_rows.extend(rows)
    episode_rows.append({"episode_id":rows[0]["episode_id"],"seed":scenario.seed,"bvr_sim_commit":"bundled-worktree",
        "bvr_sim_backend":"JSBSim-python","bvr_sim_config_hash":hashlib.sha256(json.dumps(asdict(scenario),sort_keys=True).encode()).hexdigest(),
        "jsbsim_version":getattr(jsbsim,"__version__","unknown"),"observer_aircraft_type":"F16","target_aircraft_type":"F16",
        "observer_model_version":MODEL,"target_model_version":MODEL,"initial_range_nm":scenario.range_nm,
        "initial_altitude_difference_m":scenario.altitude_difference_m,"initial_heading_difference_deg":scenario.heading_difference_deg,
        "initial_speed_difference_ms":scenario.speed_difference_ms,"initial_aspect_bin":"sampled-broad","scenario_bin":"broad",
        "observer_policy":"MAINTAIN","target_policy":"StochasticSkillManager","weapons_enabled":False,"sensor_mode":"truth",
        "episode_duration_s":DURATION_S,"termination_reason":"time_limit","skill_contract_version":skill_version,
        "skill_schedule_json":json.dumps({"switch_time_s":scenario.switch_time_s,"primary":scenario.primary_label,"secondary":scenario.secondary_label})})
    if (index + 1) % 10 == 0: print(f"completed {index + 1}/{N_FLIGHTS}")

trajectories = pd.DataFrame(trajectory_rows)
episodes = pd.DataFrame(episode_rows)
trajectories.shape, episodes.shape


In [ ]:
trajectories.columns



## 6. Validate schema, cadence, labels, leakage, and ACMI files

These are hard failures: do not persist a dataset that misses a flight, loses 10 Hz cadence, leaks a privileged field into the model allow-list, or lacks a replay.

In [ ]:
validate_columns(trajectories.columns)
assert len(episodes) == N_FLIGHTS
assert trajectories.groupby("episode_id").size().eq(round(DURATION_S / SAMPLE_DT_S) + 1).all()
intervals = trajectories.groupby("episode_id")["time_s"].diff().dropna()
assert np.allclose(intervals, SAMPLE_DT_S, atol=1e-9) and intervals.max() <= 0.1 + 1e-9
assert set(trajectories["tactical_label"].unique()) == set(TACTICAL_SKILLS)
assert trajectories[["lateral_label","vertical_label","energy_label","tactical_label"]].notna().all().all()
assert not set(MODEL_FEATURE_COLUMNS).intersection(PRIVILEGED_COLUMNS)
assert trajectories[list(MODEL_FEATURE_COLUMNS)].replace([np.inf,-np.inf],np.nan).notna().all().all()
acmi_files = sorted((OUTPUT_DIR / "acmi").glob("*.txt.acmi"))
assert len(acmi_files) == N_FLIGHTS
assert all(path.read_text(encoding="utf-8-sig").startswith("FileType=text/acmi/tacview") for path in acmi_files)
print("Validated", len(episodes), "flights,", len(trajectories), "samples and", len(acmi_files), "Tacview files")


## 7. Write canonical dataset artifacts and inspect coverage

The output contains `manifest.yaml`, `episodes.parquet`, sharded trajectories, `label_map.json`, and `acmi/*.txt.acmi`. Labels and schedules are privileged; train only from `MODEL_FEATURE_COLUMNS`. Keep splits episode/scenario grouped.

In [ ]:
try:
    simulator_commit = subprocess.check_output(["git","-C",str(BVR_SOURCE),"rev-parse","HEAD"],text=True).strip()
except (OSError, subprocess.CalledProcessError):
    simulator_commit = "bundled-worktree-unknown"
manifest = DatasetManifest(
    dataset_id="bvr_f16_1v1_jsbsim_skills_v001",
    simulator={"name":"JSBSim via BVR Sim SkillManager","commit":simulator_commit,"backend":"JSBSim-python","sample_dt_s":SAMPLE_DT_S,"integration_dt_s":SIM_DT_S},
    aircraft={"observer":"F16","target":"F16","model_version":MODEL},
    generation={"flight_count":N_FLIGHTS,"duration_s":DURATION_S,"base_seed":BASE_SEED,"scenario_sampling":"broad-stratified","manager":"StochasticSkillManager","skill_contract_version":"1.0.0"},
    fdm={"type":"JSBSim","version":getattr(jsbsim,"__version__","unknown")},
    observation={"type":"perfect-state target-centric","sensor_mode":"truth"}, weapons={"enabled":False},
)
DatasetBuilder(OUTPUT_DIR, manifest, shard_size=10).write(trajectory_rows, episode_rows)
coverage = trajectories.groupby("tactical_label").agg(flights=("episode_id","nunique"),samples=("step","size"),transitions=("transition_flag","sum"))
display(coverage)
print("Dataset:", OUTPUT_DIR.resolve())


## 8. Pre-training review

Before training, open several files from `acmi/` in Tacview, plot trajectories and labels around every switch, and reject unstable/crashed runs. Use episode-grouped splits, fit transforms on the training split only, and run persistence/rule baselines plus a shuffled-label leakage test. This pilot is deliberately small: its purpose is pipeline and label validation, not performance claims.